# Revenue Leakage & Billing Analysis

Python finance analytics project to identify outstanding payments, revenue leakage, refunds, discounts, and collection risks.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

In [ ]:
# STEP 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
# STEP 2: Load raw data
df = pd.read_csv("../data/revenue_leakage_billing_RAW_messy.csv")
df.head()

## Phase 1 — Data Understanding

In [ ]:
# STEP 3: Inspect rows, size, columns, data types, summary, missing values and duplicates
print("Shape:", df.shape)
display(df.head())
display(df.tail())
print("Columns:", df.columns.tolist())
df.info()
display(df.describe(include="all").T)
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# STEP 4: Check important categorical values
for col in ["Invoice_Status","Customer_Segment","Region","Sales_Channel","Payment_Method","Service"]:
    print("\n", col, df[col].unique())
print("Unique customers:", df["Customer_ID"].nunique())
print("Unique services:", df["Service"].nunique())

## Phase 2 — Data Cleaning

In [ ]:
# STEP 5: Create working cleaned dataset
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
df_clean["Quantity"] = df_clean["Quantity"].fillna(df_clean["Quantity"].median())
financial_cols=["Unit_Price","Discount_Percentage","Discount_Amount","Tax_Amount","Invoice_Amount","Paid_Amount","Refund_Amount","Net_Collected","Outstanding_Amount"]
for col in financial_cols:
    df_clean[col]=pd.to_numeric(df_clean[col].astype(str).str.replace(",","",regex=False).replace("nan",np.nan),errors="coerce").fillna(0)
for col in ["Customer_Segment","Region","Sales_Channel","Service","Invoice_Status","Payment_Method"]:
    df_clean[col]=df_clean[col].astype(str).str.strip().str.title()
df_clean=df_clean.drop(columns=["Currency","Customer_Status","Internal_Reference"])
print("Cleaned shape:",df_clean.shape)
display(df_clean.isnull().sum())

In [ ]:
# STEP 6: Save cleaned dataset
df_clean.to_csv("../data/revenue_leakage_billing_CLEANED.csv",index=False)

## Phase 3 — Exploratory Data Analysis

In [ ]:
# STEP 7: Overall financial KPIs
total_invoice=df_clean["Invoice_Amount"].sum()
total_net=df_clean["Net_Collected"].sum()
total_refund=df_clean["Refund_Amount"].sum()
total_outstanding=df_clean["Outstanding_Amount"].sum()
total_discount=df_clean["Discount_Amount"].sum()
collection_rate=total_net/total_invoice*100 if total_invoice else 0
outstanding_rate=total_outstanding/total_invoice*100 if total_invoice else 0
print(f"Total billed: ₹{total_invoice:,.2f}")
print(f"Net collected: ₹{total_net:,.2f}")
print(f"Outstanding: ₹{total_outstanding:,.2f}")
print(f"Refunds: ₹{total_refund:,.2f}")
print(f"Discounts: ₹{total_discount:,.2f}")
print(f"Collection rate: {collection_rate:.2f}%")
print(f"Outstanding rate: {outstanding_rate:.2f}%")

In [ ]:
# STEP 8: Status, segment, region, service and payment-method analysis
status_analysis=df_clean.groupby("Invoice_Status").agg(Invoice_Count=("Invoice_ID","count"),Total_Invoice=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Refund=("Refund_Amount","sum"),Outstanding=("Outstanding_Amount","sum")).sort_values("Outstanding",ascending=False)
segment_analysis=df_clean.groupby("Customer_Segment").agg(Invoice_Count=("Invoice_ID","count"),Total_Invoice=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Outstanding=("Outstanding_Amount","sum"),Refund=("Refund_Amount","sum"),Discount=("Discount_Amount","sum"))
segment_analysis["Collection_Rate_%"]=segment_analysis["Net_Collected"]/segment_analysis["Total_Invoice"]*100
region_analysis=df_clean.groupby("Region").agg(Invoice_Count=("Invoice_ID","count"),Total_Invoice=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Outstanding=("Outstanding_Amount","sum"),Refund=("Refund_Amount","sum"))
region_analysis["Collection_Rate_%"]=region_analysis["Net_Collected"]/region_analysis["Total_Invoice"]*100
service_analysis=df_clean.groupby("Service").agg(Invoice_Count=("Invoice_ID","count"),Total_Invoice=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Outstanding=("Outstanding_Amount","sum"),Refund=("Refund_Amount","sum"))
service_analysis["Collection_Rate_%"]=service_analysis["Net_Collected"]/service_analysis["Total_Invoice"]*100
payment_analysis=df_clean.groupby("Payment_Method").agg(Invoice_Count=("Invoice_ID","count"),Total_Invoice=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Outstanding=("Outstanding_Amount","sum"),Refund=("Refund_Amount","sum"))
payment_analysis["Collection_Rate_%"]=payment_analysis["Net_Collected"]/payment_analysis["Total_Invoice"]*100
print("STATUS"); display(status_analysis)
print("SEGMENT"); display(segment_analysis.sort_values("Outstanding",ascending=False))
print("REGION"); display(region_analysis.sort_values("Outstanding",ascending=False))
print("SERVICE"); display(service_analysis.sort_values("Outstanding",ascending=False))
print("PAYMENT METHOD"); display(payment_analysis.sort_values("Outstanding",ascending=False))

In [ ]:
# STEP 9: Calculate potential leakage and inspect top invoices
df_clean["Potential_Leakage"]=df_clean["Invoice_Amount"]-df_clean["Net_Collected"]
top_leakage=df_clean.sort_values("Potential_Leakage",ascending=False).head(10)
display(top_leakage[["Invoice_ID","Customer_Name","Invoice_Amount","Net_Collected","Outstanding_Amount","Refund_Amount","Potential_Leakage"]])

In [ ]:
# STEP 10: Core EDA charts
figs=[("Invoice Status Distribution","Invoice_Status",None,"count"),("Revenue by Customer Segment","Customer_Segment","Invoice_Amount","sum"),("Outstanding by Region","Region","Outstanding_Amount","sum"),("Revenue by Service","Service","Invoice_Amount","sum"),("Outstanding by Payment Method","Payment_Method","Outstanding_Amount","sum")]
for title,x,y,agg in figs:
    plt.figure(figsize=(9,5))
    if agg=="count": sns.countplot(data=df_clean,x=x)
    else:
        temp=df_clean.groupby(x)[y].sum().sort_values(ascending=False).reset_index()
        sns.barplot(data=temp,x=x,y=y)
    plt.title(title); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## Phase 4 — Advanced Business Analysis

In [ ]:
# STEP 11: Invoice-level collection and leakage rates
df_clean["Collection_Rate_%"]=np.where(df_clean["Invoice_Amount"]>0,df_clean["Net_Collected"]/df_clean["Invoice_Amount"]*100,0).round(2)
df_clean["Leakage_%"]=np.where(df_clean["Invoice_Amount"]>0,df_clean["Potential_Leakage"]/df_clean["Invoice_Amount"]*100,0).round(2)
def risk_category(x):
    if x < 10: return "Low Risk"
    if x <= 30: return "Medium Risk"
    if x <= 50: return "High Risk"
    return "Critical Risk"
df_clean["Leakage_Risk"]=df_clean["Leakage_%"].apply(risk_category)
display(df_clean[["Invoice_ID","Collection_Rate_%","Leakage_%","Leakage_Risk"]].head())

In [ ]:
# STEP 12: Risk distribution and financial impact
risk_financial=df_clean.groupby("Leakage_Risk").agg(Invoice_Count=("Invoice_ID","count"),Invoice_Amount=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Outstanding=("Outstanding_Amount","sum"),Potential_Leakage=("Potential_Leakage","sum"))
display(risk_financial)
high_critical=df_clean[df_clean["Leakage_Risk"].isin(["High Risk","Critical Risk"])].copy()
print("High + Critical invoices:",len(high_critical))
print(f"High + Critical leakage: ₹{high_critical['Potential_Leakage'].sum():,.2f}")

In [ ]:
# STEP 13: Segment, region, channel, discount, refund and customer analysis
segment_risk=high_critical.groupby("Customer_Segment").agg(Invoice_Count=("Invoice_ID","count"),Potential_Leakage=("Potential_Leakage","sum"),Outstanding=("Outstanding_Amount","sum")).sort_values("Potential_Leakage",ascending=False)
region_risk=high_critical.groupby("Region").agg(Invoice_Count=("Invoice_ID","count"),Potential_Leakage=("Potential_Leakage","sum"),Outstanding=("Outstanding_Amount","sum")).sort_values("Potential_Leakage",ascending=False)
channel=df_clean.groupby("Sales_Channel").agg(Invoice_Count=("Invoice_ID","count"),Total_Invoice=("Invoice_Amount","sum"),Net_Collected=("Net_Collected","sum"),Outstanding=("Outstanding_Amount","sum"),Potential_Leakage=("Potential_Leakage","sum"))
df_clean["High_Discount"]=df_clean["Discount_Percentage"]>=20
discount_impact=df_clean.groupby("High_Discount").agg(Invoice_Count=("Invoice_ID","count"),Discount=("Discount_Amount","sum"),Potential_Leakage=("Potential_Leakage","sum"),Outstanding=("Outstanding_Amount","sum"))
refund_service=df_clean.groupby("Service").agg(Total_Invoice=("Invoice_Amount","sum"),Refund=("Refund_Amount","sum"))
refund_service["Refund_Rate_%"]=refund_service["Refund"]/refund_service["Total_Invoice"]*100
top_customers=df_clean.groupby(["Customer_ID","Customer_Name"]).agg(Invoice_Count=("Invoice_ID","count"),Outstanding=("Outstanding_Amount","sum"),Potential_Leakage=("Potential_Leakage","sum")).sort_values("Potential_Leakage",ascending=False).head(10)
print("HIGH-RISK SEGMENTS"); display(segment_risk)
print("HIGH-RISK REGIONS"); display(region_risk)
print("CHANNELS"); display(channel.sort_values("Outstanding",ascending=False))
print("DISCOUNT IMPACT"); display(discount_impact)
print("REFUNDS BY SERVICE"); display(refund_service.sort_values("Refund",ascending=False))
print("TOP CUSTOMERS"); display(top_customers)

In [ ]:
# STEP 14: Collection-priority table
priority=df_clean[df_clean["Leakage_Risk"].isin(["High Risk","Critical Risk"])][["Invoice_ID","Customer_Name","Customer_Segment","Region","Service","Invoice_Amount","Net_Collected","Outstanding_Amount","Potential_Leakage","Leakage_%","Leakage_Risk"]].sort_values("Potential_Leakage",ascending=False)
display(priority.head(20))

## Phase 5 — Final Business Insights & Conclusion

In [ ]:
# STEP 15: Final KPI summary and major business findings
highest_leakage_region=df_clean.groupby("Region")["Potential_Leakage"].sum().idxmax()
highest_outstanding_segment=df_clean.groupby("Customer_Segment")["Outstanding_Amount"].sum().idxmax()
highest_leakage_service=df_clean.groupby("Service")["Potential_Leakage"].sum().idxmax()
highest_outstanding_region=df_clean.groupby("Region")["Outstanding_Amount"].sum().idxmax()
print(f"Highest leakage region: {highest_leakage_region}")
print(f"Highest outstanding segment: {highest_outstanding_segment}")
print(f"Highest leakage service: {highest_leakage_service}")
print(f"Highest outstanding region: {highest_outstanding_region}")
print(f"High + Critical invoices: {len(high_critical)}")
print(f"High + Critical leakage: ₹{high_critical['Potential_Leakage'].sum():,.2f}")

In [ ]:
# STEP 16: Final charts
for title,col,y in [("Potential Leakage by Customer Segment","Customer_Segment","Potential_Leakage"),("Outstanding Amount by Region","Region","Outstanding_Amount"),("Potential Leakage by Service","Service","Potential_Leakage")]:
    temp=df_clean.groupby(col)[y].sum().sort_values(ascending=False).reset_index()
    plt.figure(figsize=(9,5)); sns.barplot(data=temp,x=col,y=y); plt.title(title); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
# STEP 17: Save final analysis dataset
df_clean.to_csv("../data/revenue_leakage_billing_FINAL.csv",index=False)
print("Final dataset saved successfully.")

## Business Conclusion

The project identifies the gap between invoiced value and net collected revenue and prioritizes high and critical leakage-risk invoices. Customer segments, regions, services, channels, payment methods, discounts, refunds, and customers are compared to locate concentrated financial exposure.

### Recommended Actions
1. Prioritize High Risk and Critical Risk collection follow-ups.
2. Monitor customers with large outstanding balances.
3. Review regions and services with concentrated leakage.
4. Investigate high-discount invoices where collection is weak.
5. Monitor refund-heavy services.
6. Track collection rate and outstanding rate as recurring finance KPIs.

### Future Scope
- Invoice aging and due-date analysis
- Automated high-risk alerts
- Payment-delay prediction
- BI dashboard integration
- Automated finance reporting